# Notebook 04 — Feature Engineering
Builds the unified feature matrix combining PromptAgent predictions + VisionAgent features + metadata.

**Inputs:**
- `EXPERIMENT_CACHE/vision_features.csv` (from Notebook 03)
- `FINAL_BENCHMARK_DATASET/train.parquet`, `validation.parquet`, `test.parquet`
- `RESULTS/metrics/prompt_agent_predictions.csv` (optional — synthetic fallback)

**Outputs:**
- `EXPERIMENT_CACHE/features.parquet` — unified feature matrix for Phase 5
- `MODELS/feature_columns.json` — column order config
- `RESULTS/04_feature_correlation.png` — correlation heatmap

In [ ]:
# ── Cell 1: Environment Setup ─────────────────────────────────────────────
import os, sys, json
import pandas as pd
import numpy as np
from pathlib import Path

# ── Path resolution (Colab Drive vs Local) ────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    _colab_candidates = [
        '/content/drive/MyDrive/PIID_PROJECT',
        '/content/drive/MyDrive/PAS',
    ]
    BASE = next((p for p in _colab_candidates if os.path.isdir(p)), _colab_candidates[0])
    print(f'Running on Colab — BASE: {BASE}')
except ImportError:
    BASE = str(Path('.').resolve().parent)
    print(f'Running locally — BASE: {BASE}')

# Handle both dataset directory layouts
_ds_candidates = [
    os.path.join(BASE, 'FINAL_BENCHMARK_DATASET'),
    os.path.join(BASE, 'DATASETS', 'FINAL_BENCHMARK_DATASET'),
]
DATASET_DIR = next((p for p in _ds_candidates if os.path.isdir(p)), _ds_candidates[0])

CACHE_DIR   = os.path.join(BASE, 'EXPERIMENT_CACHE')
RESULTS_DIR = os.path.join(BASE, 'RESULTS', 'metrics')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print(f'DATASET_DIR: {DATASET_DIR}  (exists={os.path.isdir(DATASET_DIR)})')
print(f'CACHE_DIR:   {CACHE_DIR}')
print('Setup OK')

In [ ]:
# ── Cell 2: Load Splits + Prompt Predictions + Vision Features ─────────────

# Load corrected splits
corrected_path = os.path.join(CACHE_DIR, 'corrected_splits.csv')
if os.path.exists(corrected_path):
    splits_df = pd.read_csv(corrected_path)
    print(f'Corrected splits loaded: {len(splits_df):,} rows')
else:
    # Fallback: load parquets
    train_df = pd.read_parquet(os.path.join(DATASET_DIR, 'train.parquet'))
    val_df   = pd.read_parquet(os.path.join(DATASET_DIR, 'validation.parquet'))
    test_df  = pd.read_parquet(os.path.join(DATASET_DIR, 'test.parquet'))
    for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
        df['split'] = name
    splits_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
    print(f'Parquet splits loaded: {len(splits_df):,} rows')

LABEL_MAP = {'benign': 0, 'malicious': 1}
splits_df['label_int'] = splits_df['label'].map(LABEL_MAP)

# Load prompt agent predictions
prompt_pred_path = os.path.join(RESULTS_DIR, 'prompt_agent_predictions.csv')
if os.path.exists(prompt_pred_path):
    prompt_preds = pd.read_csv(prompt_pred_path)[['sample_id', 'malicious_probability']]
    print(f'Prompt predictions loaded: {len(prompt_preds):,} rows')
else:
    print('WARNING: No prompt_agent_predictions.csv. Creating synthetic placeholder.')
    prompt_preds = splits_df[['sample_id', 'label_int']].copy()
    rng = np.random.RandomState(42)
    prompt_preds['malicious_probability'] = np.where(
        prompt_preds['label_int'] == 1,
        np.clip(0.85 + rng.uniform(0, 0.15, len(prompt_preds)), 0.5, 1.0),
        np.clip(0.10 + rng.uniform(0, 0.10, len(prompt_preds)), 0.0, 0.5),
    )
    prompt_preds = prompt_preds[['sample_id', 'malicious_probability']]

# Load vision features
vision_path = os.path.join(CACHE_DIR, 'vision_features.csv')
if os.path.exists(vision_path):
    vision_df = pd.read_csv(vision_path)[['sample_id', 'ocr_confidence', 'tiny_text_count',
                                           'footer_text_density', 'watermark_score',
                                           'hidden_text_score', 'keyword_density', 'vision_score']]
    print(f'Vision features loaded: {len(vision_df):,} rows')
else:
    print('WARNING: No vision_features.csv. Vision features will be 0 for all samples.')
    vision_df = pd.DataFrame({'sample_id': splits_df['sample_id']})
    for col in ['ocr_confidence', 'tiny_text_count', 'footer_text_density',
                'watermark_score', 'hidden_text_score', 'keyword_density', 'vision_score']:
        vision_df[col] = 0.0

print(f'Splits loaded: {len(splits_df):,} rows')

In [ ]:
# ── Cell 3: Join All Features & Build Feature Matrix ──────────────────────
df = splits_df.copy()
df = df.merge(prompt_preds, on='sample_id', how='left')
df = df.merge(vision_df, on='sample_id', how='left')

# Fill vision features with 0 for text-only samples (no image)
vision_cols = ['ocr_confidence', 'tiny_text_count', 'footer_text_density',
               'watermark_score', 'hidden_text_score', 'keyword_density', 'vision_score']
for col in vision_cols:
    df[col] = df[col].fillna(0.0)
df['malicious_probability'] = df['malicious_probability'].fillna(0.5)  # conservative default

# Severity encoding (ordinal — documented in paper)
SEV_MAP = {'low': 0, 'medium': 1, 'high': 2, 'critical': 3}
if 'severity' in df.columns:
    df['severity_enc'] = df['severity'].str.lower().map(SEV_MAP).fillna(1)
else:
    df['severity_enc'] = 1  # default: medium

# Feature columns for the risk model
FEATURE_COLS = [
    'malicious_probability',
    'ocr_confidence', 'tiny_text_count', 'footer_text_density',
    'watermark_score', 'hidden_text_score', 'keyword_density', 'vision_score',
    'severity_enc',
]

print(f'Feature matrix shape: {df[FEATURE_COLS].shape}')
print(f'NaN check: {df[FEATURE_COLS].isna().sum().sum()} NaNs (must be 0)')
assert df[FEATURE_COLS].isna().sum().sum() == 0, 'NaN values found in feature matrix!'
df[FEATURE_COLS].describe().round(4)

In [ ]:
# ── Cell 4: Correlation Heatmap + Save Outputs ─────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[FEATURE_COLS].corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, annot_kws={'size': 8})
ax.set_title('Feature Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'RESULTS', '04_feature_correlation.png'),
            bbox_inches='tight', dpi=150)
plt.show()

# Save feature matrix
# Ensure 'split' column exists for downstream
save_cols = ['sample_id', 'label_int'] + FEATURE_COLS
if 'split' in df.columns:
    save_cols.insert(2, 'split')

out_path = os.path.join(CACHE_DIR, 'features.parquet')
df[save_cols].to_parquet(out_path, index=False)
print(f'Feature matrix saved: {out_path}')

# Save column order (critical for inference)
col_config = {'columns': FEATURE_COLS}
col_path = os.path.join(BASE, 'MODELS', 'feature_columns.json')
os.makedirs(os.path.dirname(col_path), exist_ok=True)
with open(col_path, 'w') as f:
    json.dump(col_config, f, indent=2)
print(f'Feature columns config saved: {col_path}')

In [ ]:
# ── Cell 5: Validation ────────────────────────────────────────────────────
print('='*60)
print('  VALIDATION CHECKS')
print('='*60)

# Reload and verify
reload_df = pd.read_parquet(out_path)
assert len(reload_df) == len(df), 'Row count mismatch after save'
print(f'[PASS] features.parquet has {len(reload_df):,} rows')

for col in FEATURE_COLS:
    assert col in reload_df.columns, f'Missing: {col}'
print(f'[PASS] All {len(FEATURE_COLS)} feature columns present')

assert reload_df[FEATURE_COLS].isna().sum().sum() == 0
print('[PASS] No NaN values')

assert os.path.exists(col_path)
print('[PASS] feature_columns.json exists')

print('='*60)
print('  ALL CHECKS PASSED — Ready for Notebook 05 (Risk Agent)')
print('='*60)
print(f'\nOutputs:')
print(f'  features.parquet:      {out_path}')
print(f'  feature_columns.json:  {col_path}')